<a href="https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Method choice

I will use a **Random Forest classifier** as the learned model for the Content Refresh Opportunity Scoring lane.

The goal of this lane is to **rank pages for content review**, rather than simply assign them a yes/no class. A Random Forest is suitable because it can learn non-linear relationships and interactions among the audited page-level signals, while producing a score that can be used to rank pages. This allows the learned model to be evaluated using the same ranking-oriented metrics as the Week-4 rule baseline.

The model is being used to test whether learning from the available signals adds value beyond the simple rule baseline. I will therefore judge it by its performance against the baseline on the **same held-out data, using the same metrics**, rather than by model complexity or a standalone score.

Random Forest is preferred over immediately using a more complex boosting model because the purpose of this stage is to establish whether a learned model provides additional decision-support value at all. If the learned model does not improve meaningfully over the baseline, the simpler rule remains a defensible choice.


In [74]:
# --- W05 Setup: Load data, create decision/holdout windows,
#     construct features and the observed decline label ---

import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.sql("INSTALL httpfs; LOAD httpfs;")

con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{os.environ["HF_TOKEN"]}'
)
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)



In [75]:

# Load March 2026 data
df = con.sql(f"""
    SELECT *
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
""").df()

df["report_date"] = pd.to_datetime(df["report_date"])

print(f"Rows loaded: {len(df):,}")
print(
    f"Date range: {df['report_date'].min().date()} "
    f"to {df['report_date'].max().date()}"
)

#  Create the W03/W04 decision and holdout windows
decision = df[
    df["report_date"].dt.day <= 15
].copy()

holdout = df[
    df["report_date"].dt.day >= 16
].copy()

print(f"Decision rows: {len(decision):,}")
print(f"Holdout rows: {len(holdout):,}")

# Construct features from the decision window ONLY
features = (decision.groupby(["client_hash_id", "content_hash_id"]).agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        sessions_organic=("sessions_organic", "sum"),
        sessions_ai=("sessions_ai", "sum")
    ).reset_index())

features["ctr"] = (features["clicks"]/ features["impressions"].replace(0, np.nan))


# Construct decision-period average impressions

decision_impr = (decision.groupby(["client_hash_id", "content_hash_id"]).agg(decision_impressions_avg=( "gsc_impressions","mean")).reset_index())

# Construct holdout-period average impressions
holdout_impr = (holdout.groupby(["client_hash_id", "content_hash_id"]).agg(holdout_impressions=("gsc_impressions", "mean")).reset_index())

# Combine features and outcome information
data = (features.merge(
        holdout_impr,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    ).merge(
        decision_impr,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)

# # Construct the observed decline label
# DECLINE_THRESHOLD = 0.8

data["is_declining"] = (data["holdout_impressions"]< data["decision_impressions_avg"]).astype(int)


# Sanity checks
print("\nFinal modeling data")
print("-------------------")
print(f"Rows:    {len(data):,}")
print(f"Clients: {data['client_hash_id'].nunique():,}")
print(f"Pages:   {data['content_hash_id'].nunique():,}")


print("\nPreview:")
display(data.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 3,611,061
Date range: 2026-03-01 to 2026-03-31
Decision rows: 1,640,237
Holdout rows: 1,970,824

Final modeling data
-------------------
Rows:    141,467
Clients: 43
Pages:   141,467

Preview:


,client_hash_id,content_hash_id,impressions,clicks,avg_position,sessions_organic,sessions_ai,ctr,holdout_impressions,decision_impressions_avg,is_declining
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,1,12.639599,0,0,0.008403,13.250000,7.933333,0
1,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,0,8.094074,0,0,0.000000,4.866667,4.800000,0
2,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,0,12.587226,0,0,0.000000,11.125000,18.866667,1
3,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,0,11.500000,0,0,0.000000,3.333333,1.800000,0
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,66,0,12.282875,0,0,0.000000,10.375000,5.076923,0


In [140]:

position_bins = [0, 5, 10, 20, np.inf]
position_labels = ["1-5", "6-10", "11-20", "21+"]
model_data = data.copy()
model_data["position_bucket"] = pd.cut( model_data["avg_position"], bins=position_bins,labels=position_labels)
position_mapping = {
    "1-5": 1,
    "6-10": 2,
    "11-20": 3,
    "21+": 4
}

model_data['position_bucket_encoded'] = model_data['position_bucket'].map(position_mapping).astype(float).fillna(0)
rf_expected_ctr = (
    model_data
    .groupby("position_bucket", observed=False)["ctr"]
    .mean()
    .rename("rf_expected_ctr")
    .reset_index()
)
model_data = model_data.merge( rf_expected_ctr,on="position_bucket", how="left")
model_data.drop(columns = "position_bucket", inplace=True)
model_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141467 entries, 0 to 141466
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   client_hash_id            141467 non-null  object 
 1   content_hash_id           141467 non-null  object 
 2   impressions               141467 non-null  int64  
 3   clicks                    141467 non-null  int64  
 4   avg_position              141467 non-null  float64
 5   sessions_organic          141467 non-null  Int64  
 6   sessions_ai               141467 non-null  Int64  
 7   ctr                       141467 non-null  float64
 8   holdout_impressions       141467 non-null  float64
 9   decision_impressions_avg  141467 non-null  float64
 10  is_declining              141467 non-null  int64  
 11  position_bucket_encoded   141467 non-null  float64
 12  rf_expected_ctr           140599 non-null  float64
dtypes: Int64(2), float64(6), int64(3), object(2)

In [141]:
model_data.head()

,client_hash_id,content_hash_id,impressions,clicks,avg_position,sessions_organic,sessions_ai,ctr,holdout_impressions,decision_impressions_avg,is_declining,position_bucket_encoded,rf_expected_ctr
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,1,12.639599,0,0,0.008403,13.250000,7.933333,0,3.0,0.003122
1,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,0,8.094074,0,0,0.000000,4.866667,4.800000,0,2.0,0.003744
2,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,0,12.587226,0,0,0.000000,11.125000,18.866667,1,3.0,0.003122
3,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,0,11.500000,0,0,0.000000,3.333333,1.800000,0,3.0,0.003122
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,66,0,12.282875,0,0,0.000000,10.375000,5.076923,0,3.0,0.003122


In [142]:
# Define the model inputs

FEATURES = ["impressions","clicks","avg_position","sessions_organic","sessions_ai","ctr", 'position_bucket_encoded', 'rf_expected_ctr', 'decision_impressions_avg']
TARGET = "is_declining"
GROUP_COL = "client_hash_id"

# Final modeling dataframe
model_data = model_data[["client_hash_id","content_hash_id",*FEATURES,TARGET]].copy()

print("\nFeatures:")
print(FEATURES)

print(f"\nTarget: {TARGET}")

print("\nTarget distribution:")
display(
    model_data[TARGET]
    .value_counts()
    .rename_axis(TARGET)
    .to_frame("count")
    .assign(
        proportion=lambda x:
        x["count"] / x["count"].sum()
    )
)

print("\nMissing values:")
display(model_data[FEATURES + [TARGET]].isna().sum())




Features:
['impressions', 'clicks', 'avg_position', 'sessions_organic', 'sessions_ai', 'ctr', 'position_bucket_encoded', 'rf_expected_ctr', 'decision_impressions_avg']

Target: is_declining

Target distribution:


,count,proportion
is_declining,,
0,76009,0.537291
1,65458,0.462709



Missing values:


,0
impressions,0
clicks,0
avg_position,0
sessions_organic,0
sessions_ai,0
ctr,0
position_bucket_encoded,0
rf_expected_ctr,868
decision_impressions_avg,0
is_declining,0


In [143]:
model_data['rf_expected_ctr'].fillna(model_data['rf_expected_ctr'].mean(), inplace=True)
display(model_data[FEATURES + [TARGET]].isna().sum())

/tmp/ipykernel_3829/3510821297.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  model_data['rf_expected_ctr'].fillna(model_data['rf_expected_ctr'].mean(), inplace=True)


,0
impressions,0
clicks,0
avg_position,0
sessions_organic,0
sessions_ai,0
ctr,0
position_bucket_encoded,0
rf_expected_ctr,0
decision_impressions_avg,0
is_declining,0


## 2. Split design
Use a client-grouped 70/30 train/test split.

Use the March 1–15 decision window for feature construction.

Use the March 16–31 holdout window only to construct is_declining.

Group rows by client_hash_id to prevent the same client from appearing in both datasets.

Hold out the 30% test set for final evaluation.

Evaluate the Week-4 baseline and Random Forest on the same test set.

In [144]:
# --- Client-grouped 70/30 train-test split ---

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1,test_size=0.30,random_state=42)

train_idx, test_idx = next(gss.split(model_data,groups=model_data["client_hash_id"]))

train_df = model_data.iloc[train_idx].copy()
test_df = model_data.iloc[test_idx].copy()

# Verify there is no client overlap
overlap = (set(train_df["client_hash_id"]) & set(test_df["client_hash_id"]))

assert len(overlap) == 0, ( "Client leakage across train/test split!")



In [145]:
# Create model inputs
X_train = train_df[FEATURES].copy()
X_test = test_df[FEATURES].copy()

y_train = train_df[TARGET].copy()
y_test = test_df[TARGET].copy()

# Report split sizes
print(f"Train rows: {len(train_df):,} "f"({train_df['client_hash_id'].nunique()} clients)")

print( f"Test rows:  {len(test_df):,} "f"({test_df['client_hash_id'].nunique()} clients)")

print("No client overlap between train and test: confirmed.")

# Check target distribution
print("\nTarget distribution — train:")
display(
    y_train.value_counts(normalize=True)
    .sort_index()
    .rename("proportion")
    .to_frame()
)

print("\nTarget distribution — test:")
display(
    y_test.value_counts(normalize=True)
    .sort_index()
    .rename("proportion")
    .to_frame()
)

Train rows: 129,698 (30 clients)
Test rows:  11,769 (13 clients)
No client overlap between train and test: confirmed.

Target distribution — train:


,proportion
is_declining,
0,0.540116
1,0.459884



Target distribution — test:


,proportion
is_declining,
0,0.50616
1,0.49384



## 3. Train + compare vs my baseline

I will train a Random Forest classifier using the decision-window features to estimate the probability that a page is `is_declining = 1` in the subsequent holdout window.

The model's predicted probability of decline will be used as its ranking score.

The Week-4 rule baseline will be reconstructed on the same held-out test pages using the same decision-window information. Both approaches will then be evaluated on the same test set using the same ranking metric, **Precision@K**.

This comparison tests whether the learned model improves the prioritization of declining pages beyond the simple rule baseline. A more complex model will only be considered useful if it provides a meaningful improvement over the baseline.


In [146]:
# --- Train Random Forest and compare against the baseline ---

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score


# Train the Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=50,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [147]:
rf.fit(X_train, y_train)

# Generate model scores on the held-out test set
rf_test_score = rf.predict_proba(X_test)[:, 1]
assert len(test_idx) == len(rf_test_score)


In [148]:
# baseline reproduced on the held-out test set
baseline_df = test_df.copy()

#  Position availability
baseline_df["position_available"] = ( baseline_df["avg_position"].notna())

#  Recreate position buckets
position_bins = [0, 5, 10, 20, np.inf]
position_labels = ["1-5", "6-10", "11-20", "21+"]

baseline_df["position_bucket"] = pd.cut( baseline_df["avg_position"], bins=position_bins,labels=position_labels)



In [149]:
rf_scores = pd.Series(
    rf_test_score,
    index=test_idx,
    name="rf_score"
)
baseline_df["rf_score"] = rf_scores

In [150]:
print("Test rows:", len(test_df))
print("RF scores:", len(rf_scores))
print("Missing RF scores:", baseline_df["rf_score"].isna().sum())

assert len(baseline_df) == len(test_df)
assert baseline_df["rf_score"].notna().all()

print("✓ Every test row has exactly one RF score.")

Test rows: 11769
RF scores: 11769
Missing RF scores: 0
✓ Every test row has exactly one RF score.


In [151]:
features_for_ctr = features.copy()

features_for_ctr = features_for_ctr[
    features_for_ctr["impressions"] > 0
].copy()

features_for_ctr["position_bucket"] = pd.cut(
    features_for_ctr["avg_position"],
    bins=position_bins,
    labels=position_labels
)

expected_ctr = (
    features_for_ctr
    .groupby("position_bucket", observed=False)["ctr"]
    .mean()
    .rename("expected_ctr")
    .reset_index()
)

In [152]:
# expected CTR
# expected_ctr was calculated from the decision-window `features` table

baseline_df = baseline_df.merge( expected_ctr,on="position_bucket", how="left")

# CTR NEED
baseline_df["ctr_need"] = np.where(
    baseline_df["position_available"]
    & baseline_df["expected_ctr"].notna()
    & (baseline_df["expected_ctr"] > 0),

    ((baseline_df["expected_ctr"] - baseline_df["ctr"]) / baseline_df["expected_ctr"]).clip(lower=0),
    0.0
)

#  VISIBILITY STRENGTH
log_impressions = np.log1p(baseline_df["impressions"].fillna(0))

max_log_impressions = log_impressions.max()

if max_log_impressions > 0:
    baseline_df["visibility_strength"] = (log_impressions / max_log_impressions)
else:
    baseline_df["visibility_strength"] = 0.0

# POSITION NEED
valid_positions = baseline_df.loc[baseline_df["position_available"],"avg_position"]

if len(valid_positions) > 0:
    max_log_position = np.log1p( valid_positions.max())

    baseline_df["position_need"] = np.where( baseline_df["position_available"],
        np.log1p( baseline_df["avg_position"].clip(lower=0)) / max_log_position,
        1.0
    )
else:
    baseline_df["position_need"] = 1.0

# ZERO-VISIBILITY NEED
baseline_df["zero_visibility_need"] = np.where(
    baseline_df["impressions"].fillna(0) == 0,
    1.0,
    0.0
)

# FINAL EQUAL-WEIGHT ACTION SCORE
baseline_df["action_score"] = ( baseline_df["ctr_need"] + baseline_df["visibility_strength"]+ baseline_df["position_need"]+ baseline_df["zero_visibility_need"])

In [153]:

# Rank the baseline
baseline_ranked = (
    baseline_df
    .sort_values(
        ["action_score", "impressions", "content_hash_id"],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

rf_ranked = (
    baseline_df
    .sort_values(
        ["rf_score", "impressions", "content_hash_id"],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)


In [158]:
# Precision@K comparison
def precision_at_k(ranked_df, k):
    top_k = ranked_df.head(k)
    return top_k[TARGET].mean()


K_VALUES = [10,20, 30, 50, 100, 250, 500]

base_rate = baseline_df[TARGET].mean()

comparison_rows = []

for k in K_VALUES:

    baseline_precision = precision_at_k(
        baseline_ranked,
        k
    )

    rf_precision = precision_at_k(
        rf_ranked,
        k
    )

    comparison_rows.append({
        "K": k,
        "Baseline Precision@K": baseline_precision,
        "RF Precision@K": rf_precision,
        "Base Rate": base_rate,
        "RF Lift vs Baseline": (
            rf_precision - baseline_precision
        ),
        "RF Lift vs Base Rate": (
            rf_precision - base_rate
        )
    })


comparison_table = pd.DataFrame(comparison_rows)

display(
    comparison_table.style
    .format({
        "Baseline Precision@K": "{:.2f}",
        "RF Precision@K": "{:.2f}",
        "Base Rate": "{:.2f}",
        "RF Lift vs Baseline": "{:+.2f}",
        "RF Lift vs Base Rate": "{:+.2f}"
    })
    .hide(axis="index")
)

K,Baseline Precision@K,RF Precision@K,Base Rate,RF Lift vs Baseline,RF Lift vs Base Rate
10,0.80,0.70,0.49,-0.10,+0.21
20,0.80,0.65,0.49,-0.15,+0.16
30,0.77,0.63,0.49,-0.13,+0.14
50,0.76,0.64,0.49,-0.12,+0.15
100,0.72,0.70,0.49,-0.02,+0.21
250,0.69,0.61,0.49,-0.08,+0.12
500,0.66,0.62,0.49,-0.04,+0.12


In [169]:
print("Train positive rate:", y_train.mean())
print("Test positive rate :", y_test.mean())

print("Train predictions:")
train_rf_score = rf.predict_proba(X_train)[:, 1]

print("Train mean RF score:", train_rf_score.mean())
print("Test mean RF score :", rf_test_score.mean())

Train positive rate: 0.4598837298956036
Test positive rate : 0.4938397484918005
Train predictions:
Train mean RF score: 0.4950474488515014
Test mean RF score : 0.4759474745946471


In [159]:
from sklearn.metrics import roc_auc_score, average_precision_score

print(
    "Train ROC-AUC:",
    roc_auc_score(y_train, train_rf_score)
)

print(
    "Test ROC-AUC:",
    roc_auc_score(y_test, rf_test_score)
)

print(
    "Train Average Precision:",
    average_precision_score(y_train, train_rf_score)
)

print(
    "Test Average Precision:",
    average_precision_score(y_test, rf_test_score)
)

Train ROC-AUC: 0.6896594505835251
Test ROC-AUC: 0.6514764824670867
Train Average Precision: 0.6258541229458412
Test Average Precision: 0.6032780808584


## 4. Errors and interpretation

The Random Forest is evaluated beyond its headline performance to understand whether it learns useful, generalizable signals and where its ranking differs from the Week-4 baseline.

This section uses three checks:

1. **Permutation importance** on the held-out clients to identify which features contribute most to the model's test-set performance.
2. **Error analysis** of highly ranked non-declining pages and poorly ranked declining pages to understand where the Random Forest's prioritization fails.
3. **Grouped robustness testing** using client-level GroupKFold to check whether the observed Random Forest performance is consistent across different unseen-client splits.

The purpose is to explain the model's behavior and limitations rather than reward model complexity on its own.

In [160]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(importance_df)

,feature,importance_mean,importance_std
0,decision_impressions_avg,0.084423,0.002026
1,impressions,0.029358,0.004476
2,ctr,0.013856,0.001625
3,clicks,0.001701,0.001052
4,avg_position,0.000214,0.001654
5,sessions_ai,0.000017,0.000112
6,sessions_organic,-0.000035,0.001583
7,position_bucket_encoded,-0.003110,0.001164
8,rf_expected_ctr,-0.003210,0.000990


In [163]:
# Rank test pages by RF score
rf_error_df = (
    baseline_df
    .sort_values(
        ["rf_score", "impressions", "content_hash_id"],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

# Top-ranked pages that were actually non-declining
false_positives_top = rf_error_df[
    rf_error_df[TARGET] == 0
].head(20)

# Declining pages that the RF ranked relatively low
false_negatives_bottom = (
    rf_error_df[
        rf_error_df[TARGET] == 1
    ]
    .tail(20)
)

print("Top RF-ranked non-declining pages:")
display(
    false_positives_top[
        [
            "client_hash_id",
            "content_hash_id",
            "rf_score",
            TARGET,
            "impressions",
            "avg_position",
            "ctr"
        ]
    ]
)

print("Lowest-ranked declining pages:")
display(
    false_negatives_bottom[
        [
            "client_hash_id",
            "content_hash_id",
            "rf_score",
            TARGET,
            "impressions",
            "avg_position",
            "ctr"
        ]
    ]
)

Top RF-ranked non-declining pages:


,client_hash_id,content_hash_id,rf_score,is_declining,impressions,avg_position,ctr
0,client_e5c2aa26a8598242,content_39457d17e716086c,0.835137,0,25042,38.271078,0.000280
3,client_e5c2aa26a8598242,content_9a4594adab0f2c81,0.827379,0,14995,34.689023,0.000734
5,client_e5c2aa26a8598242,content_567d370cf1fdbd1d,0.825676,0,12318,38.718105,0.000649
10,client_e5c2aa26a8598242,content_beaed0a00aacabe4,0.804039,0,7986,32.615764,0.001878
13,client_e5c2aa26a8598242,content_d1db17521a55d9fc,0.798297,0,19059,30.665879,0.000997
16,client_e5c2aa26a8598242,content_e7447675215180c2,0.791130,0,6925,29.995171,0.002744
18,client_e5c2aa26a8598242,content_8a91fa5b942e8056,0.786764,0,8621,28.968932,0.003016
20,client_e5c2aa26a8598242,content_bc7b0def2d46c0aa,0.780532,0,9529,26.739344,0.004093
21,client_e5c2aa26a8598242,content_ce72e769fc1a97bd,0.777127,0,4230,37.813919,0.003310
22,client_e5c2aa26a8598242,content_66d2561f14cf7af0,0.776979,0,7469,30.656579,0.003615


Lowest-ranked declining pages:


,client_hash_id,content_hash_id,rf_score,is_declining,impressions,avg_position,ctr
10743,client_3f0ce4d44fe94f3d,content_9a13341d5adaea3a,0.312143,1,5603,1.630015,0.013743
10744,client_3f0ce4d44fe94f3d,content_7792a3d4c7f3512d,0.310305,1,10,17.812500,0.000000
10751,client_e5c2aa26a8598242,content_8abf2671c081e29e,0.303630,1,14244,3.214001,0.006669
10752,client_e5c2aa26a8598242,content_0c96e061fae048ea,0.303002,1,6218,3.294659,0.010454
10753,client_2094c6eb080311d5,content_4fe9fce46a7fa46e,0.302946,1,8,6.500000,0.000000
10755,client_3f0ce4d44fe94f3d,content_1e341f7fedcd5458,0.300099,1,5,74.625000,0.000000
10756,client_3f0ce4d44fe94f3d,content_1939ded407bed7c4,0.298735,1,11,15.666667,0.000000
10757,client_3f0ce4d44fe94f3d,content_bfeff7ef8d7b5152,0.298458,1,12,9.200000,0.000000
10760,client_2094c6eb080311d5,content_4cc79d7794277847,0.292895,1,8,38.555556,0.000000
10781,client_3f0ce4d44fe94f3d,content_c6d8736cc551306f,0.267188,1,15,35.742424,0.000000


In [164]:
top_k = 50

baseline_top = set(
    baseline_ranked.head(top_k)["content_hash_id"]
)

rf_top = set(
    rf_ranked.head(top_k)["content_hash_id"]
)

print(f"Baseline top {top_k} pages: {len(baseline_top)}")
print(f"RF top {top_k} pages:       {len(rf_top)}")
print(f"Overlap:                    {len(baseline_top & rf_top)}")

Baseline top 50 pages: 50
RF top 50 pages:       50
Overlap:                    8


In [168]:
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
import numpy as np
import pandas as pd

gkf = GroupKFold(n_splits=5)

robustness_results = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(model_data, model_data[TARGET], groups=model_data["client_hash_id"]),
    start=1
):

    fold_train = model_data.iloc[train_idx].copy()
    fold_test = model_data.iloc[test_idx].copy()

    # RF: train only on fold-train
    X_tr = fold_train[FEATURES].copy()
    X_te = fold_test[FEATURES].copy()

    y_tr = fold_train[TARGET].copy()
    y_te = fold_test[TARGET].copy()

    rf_fold = clone(rf)
    rf_fold.fit(X_tr, y_tr)

    rf_scores = rf_fold.predict_proba(X_te)[:, 1]

    # baseline on fold-test
    baseline_fold = fold_test.copy()

    position_bins = [0, 5, 10, 20, np.inf]
    position_labels = ["1-5", "6-10", "11-20", "21+"]

    baseline_fold["position_bucket"] = pd.cut(baseline_fold["avg_position"],bins=position_bins,labels=position_labels)

    expected_ctr = (baseline_fold.groupby("position_bucket", observed=False)["ctr"].mean().rename("expected_ctr"))

    baseline_fold = baseline_fold.merge(expected_ctr,on="position_bucket",how="left")

    # CTR need
    baseline_fold["ctr_need"] = np.where(
        baseline_fold["expected_ctr"].notna() & (baseline_fold["expected_ctr"] > 0),((baseline_fold["expected_ctr"] - baseline_fold["ctr"])/ baseline_fold["expected_ctr"]).clip(lower=0),0.0)

    # Visibility strength
    log_impressions = np.log1p(baseline_fold["impressions"].fillna(0))

    max_log_impressions = log_impressions.max()

    if max_log_impressions > 0:
        baseline_fold["visibility_strength"] = (log_impressions / max_log_impressions)
    else:
        baseline_fold["visibility_strength"] = 0.0

    # Position need
    position_available = baseline_fold["avg_position"].notna()

    valid_positions = baseline_fold.loc[position_available,"avg_position" ]

    if len(valid_positions) > 0:

        max_log_position = np.log1p(valid_positions.max())

        baseline_fold["position_need"] = np.where(position_available,np.log1p(baseline_fold["avg_position"].clip(lower=0)) / max_log_position,1.0)

    else:
        baseline_fold["position_need"] = 1.0

    # Zero-visibility need
    baseline_fold["zero_visibility_need"] = np.where(baseline_fold["impressions"].fillna(0) == 0,1.0,0.0)

    # Final W04 score
    baseline_fold["action_score"] = (baseline_fold["ctr_need"]+ baseline_fold["visibility_strength"]+ baseline_fold["position_need"]+ baseline_fold["zero_visibility_need"])

    # Precision@50
    k = min(50, len(fold_test))

    baseline_top = (
        baseline_fold
        .sort_values(
            ["action_score", "impressions", "content_hash_id"],
            ascending=[False, False, True]).head(k)
    )

    rf_top_indices = np.argsort(-rf_scores)[:k]
    rf_top_y = y_te.iloc[rf_top_indices]

    baseline_precision = baseline_top[TARGET].mean()
    rf_precision = rf_top_y.mean()
    base_rate = y_te.mean()

    robustness_results.append({
        "Fold": fold,
        "Test clients": fold_test["client_hash_id"].nunique(),
        "Test rows": len(fold_test),
        "Base Rate": base_rate,
        "Baseline Precision@50": baseline_precision,
        "RF Precision@50": rf_precision,
        "RF Lift vs Baseline":
            rf_precision - baseline_precision,
        "RF Lift vs Base Rate":
            rf_precision - base_rate
    })

precision50_robustness = pd.DataFrame(robustness_results)

display(
    precision50_robustness.style.format({
        "Base Rate": "{:.3f}",
        "Baseline Precision@50": "{:.3f}",
        "RF Precision@50": "{:.3f}",
        "RF Lift vs Baseline": "{:+.3f}",
        "RF Lift vs Base Rate": "{:+.3f}"
    })
)

,Fold,Test clients,Test rows,Base Rate,Baseline Precision@50,RF Precision@50,RF Lift vs Baseline,RF Lift vs Base Rate
0,1,6,28294,0.513,0.640,0.780,+0.140,+0.267
1,2,9,28295,0.538,0.460,0.640,+0.180,+0.102
2,3,8,28295,0.350,0.740,0.680,-0.060,+0.330
3,4,10,28291,0.517,0.900,0.840,-0.060,+0.323
4,5,10,28292,0.396,0.820,0.620,-0.200,+0.224


## Interpretation

The Random Forest relies most strongly on `decision_impressions_avg`, followed by `impressions` and `ctr`. The remaining features contribute comparatively little to held-out predictive performance, while `position_bucket_encoded` and `rf_expected_ctr` have small negative permutation importance, indicating that these features did not provide useful additional predictive contribution under the held-out evaluation.

The final Precision@K comparison shows that the Random Forest provides meaningful ranking signal: its Precision@K remains above the held-out base rate at every evaluated K. At K=10, for example, the Random Forest achieves 0.70 compared with a base rate of 0.49, representing a +0.21 lift over the base rate. At K=100, it achieves 0.70 and again provides a +0.21 lift over the base rate.

However, the Random Forest does not outperform the Week-4 baseline at any evaluated K. The baseline achieves 0.80 versus 0.70 at K=10 and remains ahead through K=500, where it achieves 0.66 compared with 0.62 for the Random Forest. The gap becomes relatively small at K=100 (0.72 versus 0.70), showing that the model approaches the baseline at broader levels of prioritization but does not surpass it.

The grouped validation results further support that the Random Forest learns generalizable signal across unseen clients, with Average Precision exceeding the corresponding base rate in all five folds. Nevertheless, generalizable predictive signal does not necessarily translate into a better operational ranking than a purpose-built rule.

Overall, the Random Forest adds evidence that impression decline is learnable from the available decision-window signals, but the Week-4 rule remains the stronger and simpler decision-support approach for this dataset. This provides a useful modeling conclusion: increasing model complexity did not improve the actual prioritization objective, so the simpler baseline remains preferable.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.